In [0]:
%pip install openai
%pip install unidecode
%restart_python
%load_ext autoreload
%autoreload 2 

In [0]:
import pandas as pd
import json
import random
import openai
from openai import OpenAI
from pathlib import Path
from unidecode import unidecode

from pyspark.sql.functions import *

In [0]:
from v2_translation import *
from config import *

In [0]:
%run "./authentication script"

In [0]:
df = spark.table(INPUT_TABLE).toPandas()

In [0]:
# Test the prompt building 
#build_translation_prompt('CJ',50,'short_description50','Latin American Spanish')

In [0]:
output_path = './batch_files/batch_v0.jsonl'

# convert to jsonl format
convert_df_to_jsonl(df, TARGET_LANGUAGES, output_path)

# Upload your input file
upload = client.files.create(
    file=open(output_path, "rb"),
    purpose="batch"
)

input_file_id = upload.id
print(f"Uploaded input file: {input_file_id}")

In [0]:
# Create the batch job
batch = client.batches.create(
    input_file_id=input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h", 
    metadata={
        "description": "aso_translation_may_2025"
    }
)
print(f"Created batch job: {batch.id}")
batch_id = batch.id

In [0]:
# Check status
status = client.batches.retrieve(batch_id)
print(f"Batch status: {status.status}")
print(f"Batch id: {batch_id}")

#### Proceed once status == 'completed'

In [0]:

result_path = './results/batch_v0_results.json'

result_file_id = status.output_file_id
file_response = client.files.content(result_file_id)
result = file_response.content

# Write results to output json file
with open(result_path, "wb") as f:
    f.write(result)

In [0]:
results = []
with open(result_path, 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        results.append(json_object)

In [0]:
results[0]

In [0]:
#from utilities import *

In [0]:
#results[3]['custom_id'].split('_')

In [0]:
#results[3]['response']['body']['choices'][0]['message']['content']

In [0]:
#results[0:8]

In [0]:
#_, lang, _, row_number = results[0]['custom_id'].split('_')

In [0]:
#lang, row_number

In [0]:
LANG_MAP = dict(zip(TARGET_LANGUAGES, ['es_LA','pt_BR','it_IT','ja_JP','fr_FR','de_DE','zh_CN','ko_KR','ru_RU']))

In [0]:


#Post process the file to get the results into a decent format

json_res_parsed = []
for res in results:
    lang_full, _, row_num= res['custom_id'].split('_')
    response_status = res['response']['status_code']
    result = res['response']['body']['choices'][0]['message']['content']
    # Parse results from the response into a dict
    res_dict = {'lang': LANG_MAP[lang_full],'result':result,'row_num':row_num}

    # Append for df
    json_res_parsed.append(res_dict)

In [0]:
long_parsed_results =spark.createDataFrame(json_res_parsed) # pd.DataFrame(json_res_parsed)

In [0]:
df_holder = []

for lang_full, lang in LANG_MAP.items():
    lang_df = long_parsed_results.filter(long_parsed_results.lang == lang).select(col('row_num').cast('int'),col('result').alias(lang)).orderBy('row_num')

    df_holder.append(lang_df)

In [0]:
len(df_holder)

In [0]:
# Join all together for clear output
# annoying join, fix later
df_merged = df_holder[0].join(df_holder[1],on='row_num').join(df_holder[2],on='row_num').join(df_holder[3],on='row_num').join(df_holder[4],on='row_num').join(df_holder[5],on='row_num').join(df_holder[6],on='row_num').join(df_holder[7],on='row_num').join(df_holder[8],on='row_num')
#df_merged = df_merged

In [0]:
df_merged.display()

In [0]:
#'ds.aso_may_2025_results_merged
df_merged.write.saveAsTable('ds.aso_may_2025_results_merged_v2')

In [0]:
df_in_out_together = df.copy()

df_in_out_together['row_num'] = df_in_out_together.index

In [0]:
all_together = spark.createDataFrame(df_in_out_together).join(df_merged,on='row_num')

# old 'ds.aso_may_2025_results_in_out_together'

all_together.write.saveAsTable('ds.aso_may_2025_results_in_out_together_v2')

In [0]:
all_together.orderBy('game','char_limit').display()

In [0]:
%sql

-- select Game, char_limit, type_desc, zh_CN, ko_KR, ru_RU
--from ds.aso_may_2025_results_in_out_together_extended
--order by Game, char_limit

In [0]:
%sql

select Game, char_limit, type_desc, EN, es_LA, pt_BR, it_IT, ja_JP, fr_FR, de_DE
from ds.aso_may_2025_results_in_out_together
order by Game, char_limit

In [0]:
# Trying to join previous versions

In [0]:
%sql 

select 
  a.Game, 
  a.char_limit,
  a.type_desc,
  a.EN, 
  a.es_LA as es_LA_v1,
  b.es_LA as es_LA_v2,
  a.pt_BR as pt_BR_v1,
  b.pt_BR as pt_BR_v2,
  a.it_IT as it_IT_v1,
  b.it_IT as it_IT_v2, 
  a.ja_JP as ja_JP_v1,
  b.ja_JP as ja_JP_v2, 
  a.fr_FR as fr_FR_v1,
  b.fr_FR as fr_FR_v2,
  a.de_DE as de_DE_v1,
  b.de_DE as de_DE_v2 
from ds.aso_may_2025_results_in_out_together as a
join ds.aso_may_2025_results_in_out_together_v2 as b
on a.row_num = b.row_num and a.Game = b.Game and a.EN = b.EN
order by a.Game, a.char_limit

In [0]:
%sql 

select 
  a.Game, 
  a.char_limit,
  a.type_desc,
  a.EN, 
  a.fr_FR as fr_FR_v1,
  b.fr_FR as fr_FR_v2 
from ds.aso_may_2025_results_in_out_together as a
join ds.aso_may_2025_results_in_out_together_v2 as b
on a.row_num = b.row_num and a.Game = b.Game and a.EN = b.EN
order by a.Game, a.char_limit

In [0]:
spark.table('ds.aso_may_2025_results_in_out_together').orderBy('game','char_limit').display()